# EchoSeg-RT on a free GPU

Runtime > Change runtime type > T4 GPU, then run the cells in order. About 60 minutes end to end.

In [ ]:
REPO = 'https://github.com/YOUR_USERNAME/echoseg-rt.git'  # <- your repository
!git clone $REPO echoseg-rt
%cd echoseg-rt
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
!pytest -q

Download and preprocess CAMUS. Use `--n 150` if you are short on time.

In [ ]:
!python scripts/download_camus.py --out data/raw
!python -m echoseg.data preprocess --raw data/raw --out data/processed

Train the single-frame model, then the temporal-context model.

In [ ]:
!python -m echoseg.train

In [ ]:
!python -m echoseg.train --set data.context=2 run_name=unet_r34_ctx2

Evaluate both on the held-out test patients.

In [ ]:
!python -m echoseg.evaluate --checkpoint runs/unet_r34_ctx0/best.pt
!python -m echoseg.evaluate --checkpoint runs/unet_r34_ctx2/best.pt

In [ ]:
!python -m echoseg.benchmark --checkpoint runs/unet_r34_ctx0/best.pt

In [ ]:
!python -m echoseg.demo --checkpoint runs/unet_r34_ctx0/best.pt --out assets/demo.gif
from IPython.display import Image
Image('assets/demo.gif')

Download the reports, benchmark and GIF, then copy the numbers into README.md and MODEL_CARD.md.

In [ ]:
!zip -q results.zip assets/demo.gif runs/*/report_test.* runs/*/benchmark.md runs/*/log.csv runs/*/split.json
from google.colab import files
files.download('results.zip')